# Embedding API (preview)






<a target="_blank" href="https://colab.research.google.com/github/neo4j/graph-data-science-client/blob/main/examples/embedding-api.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

This Jupyter notebook is hosted [here](https://github.com/neo4j/graph-data-science-client/blob/main/examples/embedding-api.ipynb) in the Neo4j Graph Data Science Client Github repository.

\[embeddings description\]

This notebook will show \[...\] on a graph dataset.

In [ ]:
!uv pip install scipy torch torch-geometric pandas

In [ ]:
import os

from dotenv import load_dotenv

load_dotenv("sessions.env")

In [ ]:
from graphdatascience.session import AuraAPICredentials, CloudLocation, GdsSessions, SessionMemory

# Create a new GdsSessions object
sessions = GdsSessions(
    api_credentials=AuraAPICredentials(
        os.environ.get("CLIENT_ID"),
        os.environ.get("CLIENT_SECRET"),
        os.environ.get("PROJECT_ID"),
    )
)

gds = sessions.get_or_create(
    session_name="my_session",
    memory=SessionMemory.m_2GB,
    cloud_location=CloudLocation(provider="gcp", region="europe-west1"),
)

In [ ]:
import pandas as pd
from torch_geometric.datasets import Planetoid

dataset = Planetoid(root="datasets", name="Cora")

nodes_df = pd.DataFrame({"x": dataset.x.tolist(), "y": dataset.y})
nodes_df["nodeId"] = nodes_df.index
nodes_df["labels"] = ["Paper" for _ in range(len(nodes_df))]
rels_df = pd.DataFrame(
    {"sourceNodeId": dataset.edge_index[0], "targetNodeId": dataset.edge_index[1], "relationshipType": "CITES"}
)

In [ ]:
from graphdatascience.procedure_surface.api.node_embedding.config import FastRPConfig

with gds.graph.construct(graph_name="cora", nodes=nodes_df, relationships=rels_df) as G:
    create_result = gds.embedding.create(
        G=G,
        graph_encoder=FastRPConfig(),
        mutate_property="fastrp_embeddings",
    )

    fastrp_embeddings = gds.graph.node_properties.stream(G, node_properties="fastrp_embeddings")
fastrp_embeddings.head()

In [ ]:
from graphdatascience.procedure_surface.api.node_embedding.config import GraphSAGEConfig, MLPClassifierConfig

with gds.graph.construct(graph_name="cora", nodes=nodes_df, relationships=rels_df) as G:
    train_result = gds.embedding.train(
        G=G,
        graph_encoder=GraphSAGEConfig(target_type="Paper", out_dim=1),
        decoder=MLPClassifierConfig(),
        model_save_name="cora_model_1",
        target_label="Paper",
        target_property="y",
        feature_properties=["x"],
    )
    create_result = gds.embedding.create(
        G=G,
        graph_encoder="cora_model_1",
        feature_properties=["x"],
        mutate_property="graphsage_embeddings",
    )

    graphsage_embeddings = gds.graph.node_properties.stream(G, node_properties="graphsage_embeddings")
graphsage_embeddings.head()